# Data Scraping — `nba_api` Walkthrough

This notebook is the Phase 1 (Data Scraping) entry point for the project.

## Package overview

`nba_api` is an unofficial Python wrapper around the internal JSON endpoints that power stats.nba.com. No API key is required — it just sends HTTP requests to the same URLs the website itself uses.

It has two main areas:

- **`nba_api.stats.static`** — hardcoded, offline lookup tables (e.g. `teams.get_teams()`, `players.get_players()`). No network call, no rate-limit risk. Useful for mapping team/player IDs to names.
- **`nba_api.stats.endpoints`** — live HTTP calls to stats.nba.com (e.g. `leaguegamelog`, `teamgamelog`, `boxscoretraditionalv2`). Each endpoint is a class; instantiating it makes the request, and `.get_data_frames()` returns a list of pandas DataFrames.

A few practical notes before making live calls:

- Requests can be slow or occasionally rate-limited/blocked by stats.nba.com — it's worth setting an explicit `timeout` and being prepared to retry.
- A `season` parameter is a string like `"2023-24"`, not a single year.
- `season_type_all_star` (often just called `SeasonType`) distinguishes `"Regular Season"` from `"Playoffs"`.

In [1]:
import pandas as pd
from nba_api.stats.static import teams

# Static lookup data — no network call, just confirms the package is installed
# and shows what team metadata looks like.
nba_teams = teams.get_teams()
print(f"{len(nba_teams)} teams loaded")
pd.DataFrame(nba_teams).head()

30 teams loaded


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966


## Choosing an endpoint: `leaguegamelog`

This project's target shape (per `Claude.md`) is one row per game across the last 5 seasons. `nba_api.stats.endpoints.leaguegamelog` is the efficient way to get there: **one API call per season returns every team's game log for that season** (two rows per game — one per team). That can later be pivoted into the mirrored `_home`/`_away` row structure the project wants.

Compare that to `teamgamelog`, which only returns one team at a time — pulling a full season that way would take 30 separate calls instead of 1.

The cell below is a single smoke-test pull for one recent completed season, just to confirm the endpoint works end-to-end. It does **not** save anything to disk yet, and it is not the full 5-season loop — that's the next step once this is confirmed working.

In [2]:
from nba_api.stats.endpoints import leaguegamelog


def fetch_league_game_log(season: str, season_type: str = "Regular Season") -> pd.DataFrame:
    """Fetch one season's league-wide game log from nba_api.

    Args:
        season: Season string, e.g. "2023-24".
        season_type: "Regular Season" or "Playoffs".

    Returns:
        DataFrame with one row per team per game for that season.
    """
    # timeout guards against stats.nba.com occasionally hanging on a request
    response = leaguegamelog.LeagueGameLog(
        season=season,
        season_type_all_star=season_type,
        timeout=60,
    )
    return response.get_data_frames()[0]


# Smoke test: pull a single recent completed season only, not the full 5-season range yet.
test_df = fetch_league_game_log("2023-24")
print(test_df.shape)
test_df.head()

(2460, 29)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22023,1610612743,DEN,Denver Nuggets,0022300061,2023-10-24,DEN vs. LAL,W,240,48,...,33,42,29,9,6,12,15,119,12,1
1,22023,1610612744,GSW,Golden State Warriors,0022300062,2023-10-24,GSW vs. PHX,L,240,36,...,31,49,19,11,6,11,23,104,-4,1
2,22023,1610612747,LAL,Los Angeles Lakers,0022300061,2023-10-24,LAL @ DEN,L,240,41,...,31,44,23,5,4,12,18,107,-12,1
3,22023,1610612756,PHX,Phoenix Suns,0022300062,2023-10-24,PHX @ GSW,W,240,42,...,43,60,23,5,7,19,22,108,4,1
4,22023,1610612765,DET,Detroit Pistons,0022300068,2023-10-25,DET @ MIA,L,240,41,...,39,56,28,3,13,17,23,102,-1,1


## What's next

Once the smoke test above runs cleanly, the next pass (a separate future step) will:

1. Loop `fetch_league_game_log` over the last 5 seasons.
2. Save each season's raw output under `data/raw/` untouched (raw data is read-only per project convention).
3. Only then reshape the combined raw data into the mirrored `_home`/`_away`, one-row-per-game structure as a processing step (output goes to `data/processed/`, not `data/raw/`).